# 5.A - Corrigindo Erros e Entendendo o Ambiente

## Laboratório de diagnóstico — Databricks / Unity Catalog / Delta Lake

Este notebook reúne os testes realizados durante os Labs 01 a 05 para entender e corrigir problemas de catálogo, schema, sessões, tabelas Delta e histórico.

### Objetivos
- Entender Catalog, Schema e Table.
- Diferenciar `workspace.default` de `workspace.bookstore_lab`.
- Preferir nomes totalmente qualificados em operações críticas.
- Entender `TERMINATE` x armazenamento persistente.
- Consultar histórico Delta e usar Time Travel.
- Entender `DROP`, `UNDROP` e seus limites.

> **Regra prática:** em ambientes com vários notebooks e schemas, prefira `workspace.bookstore_lab.nome_tabela`.


## 1. Os dois laboratórios

| Objetivo | Catalog | Schema |
|---|---|---|
| Certificação | `workspace` | `default` |
| Nosso laboratório | `workspace` | `bookstore_lab` |

Mesmo nome de tabela não significa mesmo objeto:

```text
workspace.default.books
workspace.bookstore_lab.books
```


In [0]:
-- 1.1 Verifique o contexto atual
SELECT current_catalog(), current_schema();


In [0]:
-- 1.2 Tabelas da certificação
SHOW TABLES IN workspace.default;


In [0]:
-- 1.3 Tabelas do nosso laboratório
SHOW TABLES IN workspace.bookstore_lab;


## 2. Comparando os dois ambientes

O teste abaixo mostra que duas tabelas com o mesmo nome podem conter quantidades diferentes de registros.


In [0]:
SELECT 'workspace.default.books' AS tabela, COUNT(*) AS quantidade
FROM workspace.default.books
UNION ALL
SELECT 'workspace.bookstore_lab.books', COUNT(*)
FROM workspace.bookstore_lab.books;


In [0]:
SELECT 'customers' AS tabela,
       (SELECT COUNT(*) FROM workspace.default.customers) AS default_qtd,
       (SELECT COUNT(*) FROM workspace.bookstore_lab.customers) AS bookstore_lab_qtd
UNION ALL
SELECT 'orders',
       (SELECT COUNT(*) FROM workspace.default.orders),
       (SELECT COUNT(*) FROM workspace.bookstore_lab.orders)
UNION ALL
SELECT 'orders_updates',
       (SELECT COUNT(*) FROM workspace.default.orders_updates),
       (SELECT COUNT(*) FROM workspace.bookstore_lab.orders_updates);


## 3. `USE CATALOG` e `USE SCHEMA`

O nome curto depende do contexto da sessão.

```sql
USE CATALOG workspace;
USE SCHEMA bookstore_lab;
SELECT * FROM books;
```

Isso é conveniente, mas pode induzir a erros quando vários notebooks usam schemas diferentes.


In [0]:
USE CATALOG workspace;
USE SCHEMA bookstore_lab;
SELECT current_catalog(), current_schema();


In [0]:
-- Agora books significa workspace.bookstore_lab.books
SELECT COUNT(*) AS quantidade
FROM books;


## 4. Nome curto x nome totalmente qualificado

**Nome curto:**
```sql
SELECT * FROM books;
```
Depende do contexto.

**Nome totalmente qualificado:**
```sql
SELECT * FROM workspace.bookstore_lab.books;
```
Não depende do schema ativo para identificar a tabela.

### Neste laboratório, prefira o nome completo em:
- `DROP TABLE`
- `CREATE TABLE`
- `INSERT INTO`
- `MERGE INTO`
- `UPDATE`
- `DELETE`


In [0]:
-- Exemplo seguro de destino explícito. Não execute se não quiser alterar dados.
-- INSERT INTO workspace.bookstore_lab.books (...) VALUES (...);


## 5. `TERMINATE` não apaga a tabela

O compute/warehouse executa o SQL. Uma tabela Delta persistente fica separada do ciclo de vida do compute.

Se os dados parecerem diferentes depois de abrir outro notebook, primeiro verifique **Catalog + Schema + Table**.


In [0]:
-- Diagnóstico após abrir um novo notebook
SELECT current_catalog(), current_schema();
SHOW TABLES IN workspace.bookstore_lab;


## 6. Conferindo a estrutura antes do INSERT

Esse teste evita erros como o que encontramos em `orders`, quando o INSERT utilizava nomes diferentes dos definidos no `CREATE TABLE`.


In [0]:
DESCRIBE workspace.bookstore_lab.orders;


In [0]:
DESCRIBE workspace.bookstore_lab.customers;


## 7. Fotografia do estado atual

Antes de recriar ou alterar uma tabela, confira quantos registros existem.


In [0]:
SELECT 'books' AS tabela, COUNT(*) AS qtd
FROM workspace.bookstore_lab.books
UNION ALL
SELECT 'customers', COUNT(*)
FROM workspace.bookstore_lab.customers
UNION ALL
SELECT 'orders', COUNT(*)
FROM workspace.bookstore_lab.orders
UNION ALL
SELECT 'orders_updates', COUNT(*)
FROM workspace.bookstore_lab.orders_updates;


## 8. Histórico Delta

`DESCRIBE HISTORY` mostra as operações registradas na tabela Delta. Observe versões, timestamps e operações como `CREATE TABLE`, `WRITE`, `MERGE`, `UPDATE` e `DELETE`.


In [0]:
DESCRIBE HISTORY workspace.default.books;


## 9. Time Travel

No teste real, `workspace.default.books` tinha a versão 2 com 40 registros e a versão 3 com 41. Consulte versões anteriores sem alterar a tabela atual.


In [0]:
SELECT COUNT(*) AS qtd
FROM workspace.default.books VERSION AS OF 2
UNION ALL
SELECT COUNT(*)
FROM workspace.default.books VERSION AS OF 3;


In [0]:
-- Descobrir registros adicionados entre as versões
SELECT *
FROM workspace.default.books VERSION AS OF 3
EXCEPT
SELECT *
FROM workspace.default.books VERSION AS OF 2;


## 10. `DROP TABLE` e `SHOW TABLES DROPPED`

Antes de tentar recuperar uma tabela descartada, consulte o catálogo para verificar se existe uma versão recuperável.

> Este é um teste de diagnóstico; não execute DROP apenas para experimentar.


In [0]:
SHOW TABLES DROPPED IN workspace.default;


### `UNDROP`

Quando houver uma tabela descartada recuperável, o resultado pode fornecer um `tableId`. A recuperação específica pode ser feita com:

```sql
UNDROP TABLE WITH ID '<TABLE_ID>';
```

No nosso teste, `customers` foi recuperada, mas o histórico da identidade recuperada tinha apenas `CREATE TABLE` e `SET TBLPROPERTIES`; portanto ela não continha os clientes esperados.


## 11. Checklist antes de DROP / CREATE / INSERT

1. Qual é o catálogo?
2. Qual é o schema?
3. Qual é o nome completo da tabela?
4. Quantos registros existem?
5. Qual é a estrutura?
6. Existe histórico Delta?
7. Estou na certificação ou no nosso laboratório?
8. O comando usa nome totalmente qualificado?
9. Preciso mesmo de `DROP`, ou `CREATE OR REPLACE` é mais apropriado?
10. Tenho como reconstruir os dados se algo der errado?


## 12. Padrão adotado

### Certificação
```text
workspace.default
```

### Nosso laboratório
```text
workspace.bookstore_lab
```

### Exemplos
```sql
SELECT * FROM workspace.bookstore_lab.books;
INSERT INTO workspace.bookstore_lab.customers (...) VALUES (...);
MERGE INTO workspace.bookstore_lab.orders AS target ...;
```


## 13. Exercício final

**1.** Se o notebook estiver com `USE SCHEMA default`, para onde aponta `SELECT * FROM books`?

**2.** Como garantir que o SELECT consulte nosso laboratório?

**3.** `TERMINATE` apaga uma tabela Delta persistente?

**4.** Como consultar o histórico Delta?

**5.** Como verificar tabelas descartadas recuperáveis?

### Gabarito
1. `workspace.default.books`, considerando o contexto ativo.
2. `workspace.bookstore_lab.books`.
3. Não.
4. `DESCRIBE HISTORY catalog.schema.table`.
5. `SHOW TABLES DROPPED IN catalog.schema`.


## Conclusão

O principal aprendizado é que muitos problemas que parecem ser **dados perdidos** são, na realidade, problemas de **contexto**: catálogo, schema, sessão ou tabela.

> **Regra final:** quando houver dúvida, use o caminho completo: `workspace.bookstore_lab.nome_tabela`.

Antes de uma operação destrutiva, consulte o estado atual e o histórico.
